In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from torch.utils.data import DataLoader
from functools import partial


In [ ]:
cache_path = "./weights/huggingface"
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2",
    cache_dir=cache_path
).to("cuda")

In [ ]:
def find_assistant_content_sublist_indexes(l):
    # (Pdb++) processor.tokenizer.encode("<|im_start|>assistant\n")
    # [151644, 77091, 198]
    # (Pdb++) processor.tokenizer.encode("<|im_end|>\n")
    # [151645, 198]

    start_indexes = []
    end_indexes = []

    # Iterate through the list to find starting points
    for i in range(len(l) - 1):
        # Check if the current and next elements form the start sequence
        if l[i] == 151644 and l[i+1] == 77091 and l[i+2] == 198:
            start_indexes.append(i+3)
            # Now look for the first 151645 and 198 after the start
            for j in range(i+3, len(l)-1):
                if l[j] == 151645 and l[j+1] == 198:
                    end_indexes.append(j+2)
                    break  # Move to the next start after finding the end

    return list(zip(start_indexes, end_indexes))

In [ ]:
from dataset.airsim_dataset import AirSimDataset, collate_fn
from vision_process import process_vision_info
train_dataset = AirSimDataset(["../data/train_data_sample.json"], 
                                  video_frame_num = 5,
                                  target_interval = 30)

In [ ]:
batch = [train_dataset.get_batch(7)]
masks_list = []
messages_list = []
traj_folders = []
target_times = []

for data in batch:
    masks_list.append(data["prob_map"])
    message = data["prob_message"]
    messages_list.append(message)
    traj_folders.append(data["traj_dir"])
    target_times.append(data["target_time"])


In [ ]:
texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in messages_list]

In [ ]:
image_inputs, video_inputs = process_vision_info(messages_list)

In [ ]:
inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

In [ ]:
input_ids_lists = inputs['input_ids'].tolist()


In [ ]:
labels = []
for ids_list in input_ids_lists:
    label_ids = [-100] * len(ids_list) # -100 is the ignore index in loss function
    for begin_end_indexs in find_assistant_content_sublist_indexes(ids_list):
        label_ids[begin_end_indexs[0]:begin_end_indexs[1]] = ids_list[begin_end_indexs[0]:begin_end_indexs[1]]
    labels.append(label_ids)
labels = torch.tensor(labels, dtype=torch.int64)

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    collate_fn=partial(collate_fn, processor=processor, mission="prob"),
)

In [20]:
for step, batch in enumerate(train_dataloader):
    print(batch)
    if step == 0:
        break

({'pixel_values': tensor([[[[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           ...,
           [-1.0000, -0.6941,  0.1686,  ...,  0.7412,  0.8510,  0.8196],
           [-1.0000, -0.6941,  0.1765,  ...,  0.7255,  0.7882,  0.7804],
           [-1.0000, -0.6784,  0.2000,  ...,  0.7647,  0.7961,  0.7804]],

          [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           ...,
           [-1.0000, -0.7098,  0.1137,  ...,  0.6863,  0.8118,  0.7882],
           [-1.0000, -0.7176,  0.1137,  ...,  0.6706,  0.7333,  0.7333],
           [-1.0000, -0.7020,  0.1529,  ...,  0.7176,  0.7490,  0.7333]],

          [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000

In [33]:
list(batch[0].keys())

['pixel_values', 'pixel_attention_mask', 'input_ids', 'attention_mask']

In [35]:
batch[0]['pixel_attention_mask'].shape

torch.Size([1, 34, 384, 384])

In [37]:
batch[0]['input_ids'].shape

torch.Size([1, 3043])

In [38]:
batch[0]['attention_mask'].shape

torch.Size([1, 3043])

In [ ]:
# from models.processing_qwen2_vl import Qwen2VLProcessor
# from models.image_processing_qwen2_vl import Qwen2VLImageProcessor
# from models.processing_qwen2_vl import Qwen2VLProcessor
# from models.image_processing_qwen2_vl import Qwen2VLImageProcessor
# from models.pilot_llm import PilotLLM

# from PIL import Image
# import requests
# import torch
# from torchvision import io
# from typing import Dict
# from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor

# # Load the model in half-precision on the available device(s)
# modelq = Qwen2VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto",_attn_implementation="flash_attention_2", cache_dir=cache_path
# )
# processorq = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", cache_dir=cache_path)

# processorf = Qwen2VLProcessor.from_pretrained("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-VL-2B-Instruct/snapshots/895c3a49bc3fa70a340399125c650a463535e71c", padding_side="right")
# modelf = PilotLLM.from_pretrained("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-VL-2B-Instruct/snapshots/895c3a49bc3fa70a340399125c650a463535e71c")

Fetching 2 files: 100%|██████████| 2/2 [00:05<00:00,  2.55s/it]
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.64s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.05it/s]
Some weights of PilotLLM were not initialized from the model checkpoint at /storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-

In [40]:
train_dataloaderq = DataLoader(
    train_dataset,
    batch_size=1,
    collate_fn=partial(collate_fn, processor=processorq, mission="prob"),
)

In [41]:
for stepq, batchq in enumerate(train_dataloaderq):
    print(batchq)
    if step == 0:
        break

({'input_ids': tensor([[151644,   8948,    198,  ..., 151653, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]]), 'pixel_values': tensor([[-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ...,  1.5487, -1.4802, -1.4802],
        ...,
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802]]), 'image_grid_thw': tensor([[ 1, 56, 56],
        [ 1, 56, 56]])}, {'masks_list': [tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])], 'labels': tensor([[  -100,   -100,   -100,  ..., 151653, 151645,    1

In [42]:
list(batchq[0].keys())

['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw']

In [43]:
list(batch[0].keys())

['pixel_values', 'pixel_attention_mask', 'input_ids', 'attention_mask']

In [44]:
batch[0]['input_ids'].shape

torch.Size([1, 3043])

In [45]:
batchq[0]['input_ids'].shape

torch.Size([1, 1788])

In [48]:
batch[0]['attention_mask'].shape

torch.Size([1, 3043])

In [46]:
batchq[0]['attention_mask'].shape

torch.Size([1, 1788])

In [49]:
batch[0]['pixel_values'].shape

torch.Size([1, 34, 3, 384, 384])

In [50]:
batchq[0]['pixel_values'].shape

torch.Size([6272, 1176])

In [51]:
batch[0]['pixel_attention_mask'].shape

torch.Size([1, 34, 384, 384])

In [52]:
batchq[0]['image_grid_thw'].shape

torch.Size([2, 3])